<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F04_ray_on_vertex.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 04 · Ray on Vertex AI

Run the **Python-runtime models on a Ray-on-Vertex cluster** in parallel with the in-BigQuery native track — the same `main.run(cfg)` entrypoint as every other demo, dispatched to Ray by `cfg.python_runtime="ray"`. The cluster autoscales per pool by default — its initial size is a deterministic function of the run's fan-out — the job runs on it, and the cluster is torn down in a `finally` so nothing bills after the run.

> **Runs from any authenticated client** — local workstation or an in-GCP kernel. Job submission goes through the cluster's dashboard proxy host (`*.aiplatform-training.googleusercontent.com`), which serves the `JobSubmissionClient` handshake as long as the cluster is provisioned on a **PSC-I network attachment** with a **dashboard-capable head node** (`n1-standard-16`+). Both are wired by the Terraform network module and defaulted in `config` — so the older "must run inside GCP / `524` from outside" caveat no longer applies.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable. The `[ray]` extra must be installed in the kernel (`pip install -e 'scale-forecasting[ray]'`).

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[ray]"], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment + Ray infra (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses (G1). Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default.

The Ray submitter needs a little more (beyond that identity), resolved by `RayInfra`: **`SF_COMPUTE_SA`** and **`SF_CODE_BUCKET`** are required. Connectivity is picked in precedence order: **`SF_RAY_NETWORK_ATTACHMENT`** (PSC-I — the supported path) → `SF_RAY_NETWORK` (VPC peering) → unset (public). Every value comes straight from `terraform output` in `terraform/main` — `RayInfra.from_terraform_outputs()` reads them all, including `network_attachment_id`.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.ray_submit import RayInfra
from scale_forecasting.settings import Settings

settings = Settings.resolve()
infra = RayInfra.resolve()  # raises naming the first missing SF_COMPUTE_SA / SF_CODE_BUCKET
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref

# Which connectivity mode did RayInfra pick? PSC-I (network_attachment) is the supported path.
_mode = "PSC-I" if infra.network_attachment else "VPC-peering" if infra.network else "public"
print("deployment:", DATASET, "region:", settings.region)
print("ray infra: compute_sa set:", bool(infra.compute_sa), "| code_bucket:", infra.code_bucket)
print("ray connectivity:", _mode, "| attachment:", infra.network_attachment or "(none)")

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — `compute_engine` splits Ray / BigQuery / ensemble — and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=12, pause=3.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Parameters — edit me

Everything that shapes the run is set here as plain Python, then assembled into a **`RunConfig`** — the one frozen object that drives the run and is logged verbatim to `run_registry.raw_config`, so *this cell is the experiment record* (G2/G3). No separate JSON file to open.

This run mixes both tracks under **one `run_id`**: the stats models run on **Ray** (`compute_engine='ray'`) while the natives run in **BigQuery** (`compute_engine='bigquery'`), in parallel — the Ray analogue of notebook 03's Spark ∥ BigQuery.

> **Coming from PySpark + pandas UDFs?** This is the *same* per-`(series, model)` unit of work you'd run in an `applyInPandas` fan-out — `worker.run_cell` is identical on Spark and Ray (G1) — and it reads the **same** `source_series_iceberg` input. The difference: no Spark session to stand up or tune, Ray ∥ BigQuery under one `run_id`, and one Ray cluster can host many frameworks (including Spark workloads via RayDP). The engine reads the source panel two ways, chosen by `compute.ray_read_mode`: the default `driver_collect` (the BigQuery Storage Read API client) and the opt-in `ray_data` (`ray.data.read_bigquery`) — both go through the same Storage Read API, which is what makes the Iceberg/native input transparent. *Next steps (not yet shipped):* distributing the `ray.data` read as blocks straight into the fan-out (skipping the driver round-trip) and Spark-on-Ray (RayDP).

- **`RUN_NAME`** carries a timestamp so each execution is its own clean run (the cell tables are append-only, so a fresh `run_id` avoids overwriting a still-buffering prior run).
- **`SOURCE_TABLE`** defaults to `source_series_iceberg` to *demonstrate Ray reading the managed-Iceberg input*; the example also ships as `source_series_native` (identical series) so you can flip the suffix and compare storage formats on the same run shape.
- **`RAY_MODELS`** run on the cluster; **`BQ_MODELS`** run in BigQuery. The shipped example is univariate; the generic exog seam stays available for your own data (set `features.exog` + carry the column in your source table).
- **`RAY_READ_MODE`** picks the driver's source reader: `"driver_collect"` (default, the proven Storage Read client) or `"ray_data"` (the Ray-native `ray.data.read_bigquery`). Both read the same table over the same Storage Read API and hand the fan-out an identical panel — the knob just selects the client.
- **`USE_GPU=False`** → `plan_cluster` sizes the GPU pool to **zero** (CPU workers only, no T4 quota needed). Flip to `True` and add `"neuralprophet"` to `RAY_MODELS` for the fractional-T4 showpiece — identical call, one flag apart (needs `NVIDIA_T4_GPUS` quota in a `RAY_REGIONS` region).

> Expect **~15–25 min**: cluster stand-up dominates. `RAY_REGIONS` lets the launcher hop US regions if one transiently stocks out on capacity.

In [ ]:
from scale_forecasting.config import RunConfig
from scale_forecasting.registry.ids import make_run_id

# === Parameters — edit me ===============================================
RUN_NAME = f"nb04 ray+bq {int(time.time())}"  # timestamped → a fresh run each execution
SOURCE_TABLE = (
    "source_series_iceberg"  # Ray reads the SAME Iceberg input; _native to compare storage
)
RAY_MODELS = ["theta", "holtwinters"]  # → run on the Ray cluster
BQ_MODELS = ["arima_plus", "timesfm"]  # → run in BigQuery, in parallel
HORIZON = 28
SERIES_LIMIT = 100  # the demo scale (same first 100 series every approach uses)
HOLIDAYS = ["US"]
USE_GPU = False  # True + "neuralprophet" in RAY_MODELS → T4 showpiece
RAY_READ_MODE = "driver_collect"  # or "ray_data" (ray.data.read_bigquery) — same Storage Read API
BACKTEST = True  # OOF metric panel comparable across both engines
N_FOLDS = 2
RAY_REGIONS = ["us-central1", "us-east1", "us-west1"]
# ========================================================================

cfg = RunConfig(
    run_name=RUN_NAME,
    python_runtime="ray",  # dispatch the Python models to Ray (not Spark)
    data={"source_table": SOURCE_TABLE, "horizon": HORIZON, "series_limit": SERIES_LIMIT},
    models=[*RAY_MODELS, *BQ_MODELS],  # router splits these by runtime automatically
    features={"holidays": HOLIDAYS},
    backtest={"enabled": BACKTEST, "n_folds": N_FOLDS, "horizon": HORIZON, "step": HORIZON},
    compute={"use_gpu": USE_GPU, "ray_regions": RAY_REGIONS, "ray_read_mode": RAY_READ_MODE},
)
run_id = make_run_id(cfg)
print("run_id:", run_id)
print("ray models:", RAY_MODELS, "| bq models:", BQ_MODELS, "| runtime:", cfg.python_runtime)

## Run — Ray ∥ BigQuery, one `run_id`

One call. `main.run(cfg)` sizes + creates the Ray cluster on the PSC-I attachment — autoscaling per pool by default, its initial size a deterministic function of the fan-out — runs `RAY_MODELS` on it **in parallel** with `BQ_MODELS` in BigQuery under the shared `run_id`, then tears the cluster down in a `finally`. It returns the same `run_id` we computed above.

In [ ]:
from scale_forecasting import main

returned = main.run(cfg)
assert returned == run_id
print("ray ∥ bigquery run complete:", run_id)

## Review — both engines on one leaderboard

The stats models ran on Ray (`compute_engine='ray'`); the natives in BigQuery (`compute_engine='bigquery'`) — all under one `run_id`, ranked by `mean_wape`.

In [ ]:
leaderboard(run_id, expect_models=cfg.models)

In [ ]:
run_summary(run_id)

## What the run recorded — the cluster sizing

The header's `job_telemetry` audits the sizing decision that actually ran: `runtime='ray'`, the autoscale spec (per-pool min/max) with the deterministic initial per-pool node counts, and (on the GPU path) the calibrated `sizing_gpu_fraction` + `accelerator_type`. On this CPU-only run `gpu_node_count` is `0` — the whole cluster is the CPU worker pool.

In [ ]:
import json

hdr = _query(
    f"SELECT TO_JSON_STRING(job_telemetry) AS job_telemetry "
    f"FROM `{DATASET}.run_registry` WHERE run_id=@run_id",
    run_id,
)
json.loads(hdr["job_telemetry"].iloc[0]) if not hdr.empty else "(header not visible yet)"